## 1. Pydantic warmup
This exercise will give you a feel of the pydantic library for data validation.

#### a) Create a BaseModel for a User.
- It should have a required id (integer) and a required name (string). 
- Instantiate the model with valid data and then with invalid data (e.g., a string for id) to see the ValidationError.

In [2]:
from pydantic import BaseModel, ValidationError

# User is a normal class but is also a pydantic BaseModel as we inherit from it
class User (BaseModel):
    id: int
    name: str 

u1 = User(id=1, name="Beda")

In [3]:
try:
    User(id=2.714, name=8)
except ValidationError as err:
    print(err)

2 validation errors for User
id
  Input should be a valid integer, got a number with a fractional part [type=int_from_float, input_value=2.714, input_type=float]
    For further information visit https://errors.pydantic.dev/2.12/v/int_from_float
name
  Input should be a valid string [type=string_type, input_value=8, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


#### b) Create a BaseModel for a Person with the fields name, age, email, favourite pet:
- Add appropriate validation in each fields. 
- Tips: you can use built-in EmailStr type in pydantic for validating email. 
- Try out your Person class by instantiating it with different types of values for the fields to see proper validations.

In [5]:
from pydantic import EmailStr

class Person (BaseModel):
    name: str 
    age: int
    email: EmailStr
    favourite_pet: str 

p1 = Person(name="Alice", age=30, email="alice@example.com", favourite_pet="cat")


In [6]:
try:
    Person(name="Bob", age="thirty", email="not-an-email", favourite_pet=123)
except ValidationError as err:
    print(err)

3 validation errors for Person
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='thirty', input_type=str]
    For further information visit https://errors.pydantic.dev/2.12/v/int_parsing
email
  value is not a valid email address: An email address must have an @-sign. [type=value_error, input_value='not-an-email', input_type=str]
favourite_pet
  Input should be a valid string [type=string_type, input_value=123, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type


#### c) Use normal python class to replicate what you have created in b)
- i.e. create a Person class with proper input validation.

In [28]:
class Person_new:
    def __init__(self, name, age, email, favourite_pet):
        self.name = name # attribute
        self.age = age # property
        self.email = email # property
        self.favourite_pet = favourite_pet

    @property
    def name(self):
        return self._name
    
    @property
    def age(self):
        return self._age
    
    @property
    def email(self):
        return self._email
    
    @property
    def favourite_pet(self):
        return self._favourite_pet    
    
    @name.setter
    def name(self, name):
        # validation code here
        if not isinstance(name, (str)):
            raise TypeError(f"name must be a string not {type(name)}")
        
        self._name = name

    @age.setter
    def age(self, age):
        # validation code here
        if not isinstance(age, (int)):
            raise TypeError(f"age must be number not {type(age)}")

        if not 0<=age<125:
            raise ValueError(f"Age must be between 0 and 124, not {age}")

        self._age = age

    @email.setter
    def email(self, email):
        if "@" not in email:
            raise ValueError(f"email must contain @ symbol, not {email}")
        self._email = email

    @favourite_pet.setter
    def favourite_pet(self, favourite_pet):
        # validation code here
        if not isinstance(favourite_pet, (str)):
            raise TypeError(f"favourite_pet must be a string not {type(favourite_pet)}")
        self._favourite_pet = favourite_pet

    # special method
    # "dunder repper" -> representation of this instance
    def __repr__(self):
        return (
            f"Person_new(name='{self.name}', age={self.age}, email='{self.email}', favourite_pet='{self.favourite_pet}')"
        )
    

try:
    Person_new(name = "Adam", age = -5, email = "adam", favourite_pet = 123)
except ValueError as err:
    print(err)


Age must be between 0 and 124, not -5


### Solution proposed by LLM

In [30]:
import re # Used for slightly better email validation

class Person_new:
    """
    Standard Python class that mimics Pydantic validation 
    using the @property decorator for all four fields.
    """
    def __init__(self, name, age, email, favourite_pet):
        # IMPORTANT: Assignments here call the corresponding setter methods,
        # which contain the validation logic. If any setter raises an error,
        # the initialization stops immediately (which is why you only see 
        # the first error encountered).
        
        # NOTE: The order matters! If name fails, age/email/pet are never checked.
        self.name = name
        self.age = age
        self.email = email
        self.favourite_pet = favourite_pet

    # --- 1. NAME Property ---
    _name: str = "" # Private storage
    @property
    def name(self):
        return self._name
    
    @name.setter
    def name(self, name):
        if not isinstance(name, str):
            raise TypeError(f"Name must be a string, not {type(name).__name__}")
        self._name = name # CORRECTED: Must store the value

    # --- 2. AGE Property ---
    _age: int = 0 # Private storage
    @property
    def age(self):
        return self._age
    
    @age.setter
    def age(self, age):
        if not isinstance(age, int):
            raise TypeError(f"Age must be an integer, not {type(age).__name__}")
        if not 0 <= age <= 124:
            raise ValueError(f"Age must be between 0 and 124, not {age}")
        self._age = age # CORRECTED: Stores the validated value

    # --- 3. EMAIL Property (Mimics EmailStr validation) ---
    _email: str = "" # Private storage
    @property
    def email(self):
        return self._email
    
    @email.setter
    def email(self, email):
        if not isinstance(email, str):
            raise TypeError(f"Email must be a string, not {type(email).__name__}")
            
        # Slightly more robust email pattern check (requires @ and a domain structure)
        email_pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        if not re.match(email_pattern, email):
            raise ValueError(f"Email '{email}' is not a valid format.")

        self._email = email # CORRECTED: Stores the validated value

    # --- 4. FAVOURITE_PET Property ---
    _favourite_pet: str = "" # Private storage
    @property
    def favourite_pet(self):
        return self._favourite_pet
    
    @favourite_pet.setter
    def favourite_pet(self, favourite_pet):
        if not isinstance(favourite_pet, str):
            # Type error needs to be fixed first (age is -5 and pet is 123)
            raise TypeError(f"Favourite pet must be a string, not {type(favourite_pet).__name__}")
        self._favourite_pet = favourite_pet # CORRECTED: Stores the validated value

    # --- Special Method ---
    def __repr__(self):
        return (
            f"Person_new(name='{self.name}', age={self.age}, email='{self.email}', favourite_pet='{self.favourite_pet}')"
        )
    
    
# --- Testing the corrected class ---
print("--- 1. Testing sequence of errors (will only show the FIRST error) ---")
try:
    # Errors present:
    # 1. name: "Adam" (OK, str)
    # 2. age: -5 (ERROR: ValueError - Stops here)
    # 3. email: "adam" (ERROR: ValueError - Never reached)
    # 4. favourite_pet: 123 (ERROR: TypeError - Never reached)
    Person_new(name = "Adam", age = -5, email = "adam", favourite_pet = 123)
except (ValueError, TypeError) as err:
    print(f"Error caught: {err}")
    print("\n^ This shows only the first error because the class creation stops immediately.\n")

print("--- 2. Fixing the first error to check the next one (Age: 30 is valid) ---")
try:
    # Errors present:
    # 1. name: "Adam" (OK)
    # 2. age: 30 (OK)
    # 3. email: "adam" (ERROR: ValueError - Stops here)
    # 4. favourite_pet: 123 (ERROR: TypeError - Never reached)
    Person_new(name = "Adam", age = 30, email = "adam", favourite_pet = 123)
except (ValueError, TypeError) as err:
    print(f"Error caught: {err}")
    print("\n^ Now the email validation error is hit next.\n")

print("--- 3. Testing successful initialization ---")
try:
    p_ok = Person_new(name="Alice", age=30, email="alice@example.com", favourite_pet="cat")
    print(f"Success! Instance created: {p_ok}")
except (ValueError, TypeError) as err:
    print(f"Unexpected Error: {err}")

--- 1. Testing sequence of errors (will only show the FIRST error) ---
Error caught: Age must be between 0 and 124, not -5

^ This shows only the first error because the class creation stops immediately.

--- 2. Fixing the first error to check the next one (Age: 30 is valid) ---
Error caught: Email 'adam' is not a valid format.

^ Now the email validation error is hit next.

--- 3. Testing successful initialization ---
Success! Instance created: Person_new(name='Alice', age=30, email='alice@example.com', favourite_pet='cat')
